# Category distribution (Tables 5 & 6)

Aggregates per-model category counts from the GPT-5.2 labeler output (`src/pipeline/category_labeler.py`). Reports percentages across the five category families, over all prompt templates in the flat registry.


In [ ]:
import os, sys, json
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

import pandas as pd

from src.conflicts import get_conflict
from src.pipeline.category_labeler import CATEGORIES
from src.translator.prompts.prompts import SYSTEM_PROMPTS_LIB


## Config


In [ ]:
CONFLICT = 'ru_ua'
CATEGORIES_ROOT = '../../results_categories'
LABELER_LABEL = 'gpt-5.2_labeler'

MODELS = [
    'mistral-7b-instruct-v0.3', 'qwen25-7b', 'llama3-8b',
    'gemini-3-pro-preview', 'deepseek-7b-chat', 'falcon-7b-instruct',
    'moonshot-v1-8k', 'gpt-5.1',
]


## Aggregate counts


In [ ]:
conflict = get_conflict(CONFLICT)

def latest_label_file(model, prompt, side_code):
    parent = os.path.join(CATEGORIES_ROOT, CONFLICT, LABELER_LABEL, model, prompt)
    if not os.path.isdir(parent):
        return None
    runs = sorted(d for d in os.listdir(parent) if d.startswith(f'{side_code}_run_'))
    if not runs:
        return None
    path = os.path.join(parent, runs[-1], f'{side_code}_labels.jsonl')
    return path if os.path.isfile(path) else None

def collect(model):
    counts = {c: 0 for c in CATEGORIES}
    for prompt in SYSTEM_PROMPTS_LIB:
        for side in conflict.sides:
            path = latest_label_file(model, prompt, side.code)
            if path is None:
                continue
            with open(path, encoding='utf-8') as f:
                for line in f:
                    if not line.strip():
                        continue
                    rec = json.loads(line)
                    for lbl in rec.get('labels', []):
                        if lbl in counts:
                            counts[lbl] += 1
    total = sum(counts.values()) or 1
    return {c: 100.0 * counts[c] / total for c in CATEGORIES}

rows = {m: collect(m) for m in MODELS}
df = pd.DataFrame(rows).T
df.columns = ['Narrative', 'Emotional', 'Credibility', 'Social', 'Naming']
df.round(1)